# Wyscout Player Advanced Stats → Formatted Excel
Fetches advanced stats for every player found in local event JSON files and exports a formatted `.xlsx`.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install / upgrade openpyxl if needed
!pip install -q --upgrade openpyxl

In [ ]:
import os
import json
import time
import requests
import pandas as pd
from pathlib import Path
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
API_KEY          = "neom-9172-sjou-mzxw"
BASE_URL         = "https://api.boxtobox.ai/v1/wy/api/v3"

COMPETITION_WYID = 364
SEASON_ID        = 191622
SLEEP_SECONDS    = 0.15

EVENTS_DIR       = Path("/content/drive/MyDrive/Event data/Wyscout/Premier League/2025-2026")
OUT_XLSX         = "/content/drive/MyDrive/Event data/Wyscout/epl.xlsx"
COMPETITIONS_CSV = "/content/drive/MyDrive/Event data/Wyscout/domestic_competitions_all_areas.csv"

In [ ]:
# ── API helper ───────────────────────────────────────────────────────────────
def get_player_advanced_stats(player_id: int, comp_id: int, season_id: int | None):
    params = {"api_key": API_KEY, "compId": comp_id}
    if season_id is not None:
        params["seasonId"] = season_id
    r = requests.get(
        f"{BASE_URL}/players/{player_id}/advancedstats",
        params=params,
        timeout=30,
    )
    r.raise_for_status()
    return r.json()

In [ ]:
# ── Position formatter ───────────────────────────────────────────────────────
def format_positions(position_list):
    if not isinstance(position_list, list):
        return None
    sorted_pos = sorted(position_list, key=lambda x: x.get("percent", 0), reverse=True)
    codes = [
        p.get("position", {}).get("code", "").upper()
        for p in sorted_pos
        if p.get("position", {}).get("code")
    ]
    return ",".join(codes) if codes else None

In [ ]:
# ── Build player lookup from event JSONs ─────────────────────────────────────
def build_player_lookup(events_dir: Path) -> pd.DataFrame:
    rows = []
    for fp in events_dir.glob("*.json"):
        try:
            with open(fp, "r", encoding="utf-8") as f:
                data = json.load(f)
            for ev in data.get("events", []):
                player = ev.get("player", {})
                team   = ev.get("team", {})
                pid    = player.get("id")
                if pid is None:
                    continue
                rows.append({
                    "playerId":   int(pid),
                    "playerName": player.get("name"),
                    "teamName":   team.get("name"),
                })
        except Exception:
            continue

    df = pd.DataFrame(rows)

    name_df = (
        df.groupby(["playerId", "playerName"]).size().reset_index(name="n")
        .sort_values(["playerId", "n"], ascending=[True, False])
        .drop_duplicates("playerId")[["playerId", "playerName"]]
    )
    team_df = (
        df.groupby(["playerId", "teamName"]).size().reset_index(name="n")
        .sort_values(["playerId", "n"], ascending=[True, False])
        .drop_duplicates("playerId")[["playerId", "teamName"]]
    )
    return name_df.merge(team_df, on="playerId", how="left")

In [ ]:
# ── Add competition name from CSV ─────────────────────────────────────────────
def add_competition_name(df_final: pd.DataFrame, competitions_csv_path: str) -> pd.DataFrame:
    comps = pd.read_csv(competitions_csv_path)

    comp_id_col = next((c for c in ["competitionId", "wyId", "id"] if c in comps.columns), None)
    if comp_id_col is None:
        raise RuntimeError(f"No competition id column found. Columns: {comps.columns.tolist()}")
    if "name" not in comps.columns:
        raise RuntimeError(f"No 'name' column found. Columns: {comps.columns.tolist()}")

    comps = comps.rename(columns={comp_id_col: "competitionId", "name": "competitionName"})
    comps["competitionId"] = pd.to_numeric(comps["competitionId"], errors="coerce")

    df_comp_col = next(
        (c for c in ["competitionId", "competition.id", "compId"] if c in df_final.columns), None
    )
    if df_comp_col is None:
        df_final = df_final.copy()
        df_final["competitionId"] = COMPETITION_WYID
        df_comp_col = "competitionId"

    df_final = df_final.copy()
    df_final["competitionId"] = pd.to_numeric(df_final[df_comp_col], errors="coerce")

    extra = [c for c in ["area.name", "areaName", "country", "countryName"] if c in comps.columns]
    comps_small = comps[["competitionId", "competitionName"] + extra].drop_duplicates("competitionId")

    return df_final.merge(comps_small, on="competitionId", how="left")

In [ ]:
# ── Excel formatting helpers ──────────────────────────────────────────────────
HEADER_BG     = "1F3864"
HEADER_FG     = "FFFFFF"
ID_BG         = "D6E4F0"
ALT_ROW_BG    = "F2F7FB"
BORDER_COL    = "BDD7EE"
IDENTITY_COLS = {"playerId", "playerName", "teamName", "positions_codes",
                 "competitionId", "competitionName"}

THIN        = Side(style="thin", color=BORDER_COL)
THIN_BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)


def _apply_style(cell, style: dict):
    for attr, val in style.items():
        setattr(cell, attr, val)


def _col_width(series: pd.Series, header: str, max_width: int = 30) -> float:
    max_len = max(
        len(str(header)),
        series.dropna().astype(str).str.len().max() if not series.dropna().empty else 0,
    )
    return min(max_len + 2, max_width)


def _number_format(col_name: str, series: pd.Series) -> str:
    low = col_name.lower()
    if "id" in low:
        return "0"
    if any(k in low for k in ("percent", "pct", "rate", "%", "ratio", "avg", "average", "mean")):
        return "0.00"
    if pd.api.types.is_numeric_dtype(series):
        if series.dropna().apply(float.is_integer).all() if not series.dropna().empty else True:
            return "0"
        return "0.00"
    return "General"


def format_stats_sheet(ws, df: pd.DataFrame, sheet_title: str = "Advanced_Stats"):
    cols   = list(df.columns)
    n_rows = len(df)
    n_cols = len(cols)

    # Header
    for ci, col in enumerate(cols, 1):
        cell = ws.cell(row=1, column=ci, value=col)
        _apply_style(cell, {
            "font":      Font(name="Calibri", bold=True, color=HEADER_FG, size=11),
            "fill":      PatternFill("solid", fgColor=HEADER_BG),
            "alignment": Alignment(horizontal="center", vertical="center", wrap_text=True),
            "border":    THIN_BORDER,
        })

    # Data rows
    for ri, (_, row) in enumerate(df.iterrows(), 2):
        alt = (ri % 2 == 0)
        for ci, col in enumerate(cols, 1):
            val = row[col]
            if hasattr(val, "item"):
                val = val.item()
            cell = ws.cell(row=ri, column=ci, value=val)
            if col in IDENTITY_COLS:
                _apply_style(cell, {
                    "fill":      PatternFill("solid", fgColor=ID_BG),
                    "font":      Font(name="Calibri", size=10,
                                     bold=(col == "playerName")),
                    "alignment": Alignment(horizontal="left", vertical="center"),
                    "border":    THIN_BORDER,
                })
            else:
                bg = ALT_ROW_BG if alt else "FFFFFF"
                _apply_style(cell, {
                    "fill":      PatternFill("solid", fgColor=bg),
                    "font":      Font(name="Calibri", size=10),
                    "alignment": Alignment(horizontal="right", vertical="center"),
                    "border":    THIN_BORDER,
                })
                if pd.api.types.is_numeric_dtype(df[col]):
                    cell.number_format = _number_format(col, df[col])

    # Column widths
    for ci, col in enumerate(cols, 1):
        ws.column_dimensions[get_column_letter(ci)].width = _col_width(df[col], col)

    # Freeze panes (header + identity cols)
    id_count = sum(1 for c in cols if c in IDENTITY_COLS)
    ws.freeze_panes = f"{get_column_letter(id_count + 1)}2"

    # Excel Table with filter dropdowns
    if n_rows > 0:
        tbl = Table(
            displayName=sheet_title.replace(" ", "_"),
            ref=f"A1:{get_column_letter(n_cols)}{n_rows + 1}",
        )
        tbl.tableStyleInfo = TableStyleInfo(
            name="TableStyleMedium9",
            showFirstColumn=False, showLastColumn=False,
            showRowStripes=True,   showColumnStripes=False,
        )
        ws.add_table(tbl)

    # Row heights
    ws.row_dimensions[1].height = 36
    for r in range(2, n_rows + 2):
        ws.row_dimensions[r].height = 16

    ws.title = sheet_title


def format_failures_sheet(ws, df: pd.DataFrame):
    for ci, col in enumerate(df.columns, 1):
        cell = ws.cell(row=1, column=ci, value=col)
        _apply_style(cell, {
            "font":      Font(name="Calibri", bold=True, color=HEADER_FG, size=11),
            "fill":      PatternFill("solid", fgColor=HEADER_BG),
            "alignment": Alignment(horizontal="center", vertical="center"),
            "border":    THIN_BORDER,
        })
    for ri, (_, row) in enumerate(df.iterrows(), 2):
        for ci, col in enumerate(df.columns, 1):
            val = row[col]
            if hasattr(val, "item"):
                val = val.item()
            cell = ws.cell(row=ri, column=ci, value=val)
            _apply_style(cell, {
                "fill":   PatternFill("solid", fgColor="FFF2CC"),
                "font":   Font(name="Calibri", size=10),
                "border": THIN_BORDER,
            })
    for ci, col in enumerate(df.columns, 1):
        ws.column_dimensions[get_column_letter(ci)].width = _col_width(df[col], col)
    ws.freeze_panes = "A2"
    ws.title = "Failures"

In [ ]:
# ── Fetch stats ───────────────────────────────────────────────────────────────
lookup     = build_player_lookup(EVENTS_DIR)
player_ids = sorted(lookup["playerId"].unique())

rows     = []
failures = []

for i, pid in enumerate(player_ids, 1):
    print(f"[{i}/{len(player_ids)}] player {pid}")
    try:
        data = get_player_advanced_stats(pid, comp_id=COMPETITION_WYID, season_id=SEASON_ID)
        if isinstance(data, dict) and "error" in data:
            failures.append({"playerId": pid, "error": data["error"]})
            continue
        flat = pd.json_normalize(data, sep=".")
        flat["playerId"]       = pid
        flat["positions_codes"] = format_positions(data.get("positions")) if isinstance(data, dict) else None
        rows.append(flat)
    except Exception as e:
        failures.append({"playerId": pid, "error": str(e)})
    time.sleep(SLEEP_SECONDS)

df_stats = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
df_fail  = pd.DataFrame(failures)
print(f"\nFetched {len(rows)} players  |  {len(failures)} failures")

In [ ]:
# ── Merge & column order ──────────────────────────────────────────────────────
df_final = lookup.merge(df_stats, on="playerId", how="left")
df_final = add_competition_name(df_final, COMPETITIONS_CSV)

front    = [c for c in ["playerId", "playerName", "teamName", "positions_codes",
                         "competitionId", "competitionName"] if c in df_final.columns]
stat_cols = sorted([c for c in df_final.columns if c not in front])
df_final  = df_final[front + stat_cols]

print(f"Shape: {df_final.shape}")
df_final.head()

In [ ]:
# ── Write Excel & apply formatting ───────────────────────────────────────────
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    df_final.to_excel(writer, sheet_name="advancedstats", index=False)
    if not df_fail.empty:
        df_fail.to_excel(writer, sheet_name="failures", index=False)

wb = load_workbook(OUT_XLSX)
format_stats_sheet(wb["advancedstats"], df_final, sheet_title="Advanced_Stats")
if not df_fail.empty and "failures" in wb.sheetnames:
    format_failures_sheet(wb["failures"], df_fail)
wb.save(OUT_XLSX)

print("✅ Saved formatted Excel to:", OUT_XLSX)